#Init

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Silver Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, T.StringType):
        trimmed_col = F.trim(F.col(field.name))
        
        df = df.withColumn(
            field.name, 
            F.when(trimmed_col == "", F.lit(None).cast("string"))
             .otherwise(trimmed_col)
        )

##Cleaning Dates

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (F.col("sls_order_dt") == 0) | (F.length(F.col("sls_order_dt")) != 8), None)
        .otherwise(F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (F.col("sls_ship_dt") == 0) | (F.length(F.col("sls_ship_dt")) != 8), None)
        .otherwise(F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (F.col("sls_due_dt") == 0) | (F.length(F.col("sls_due_dt")) != 8), None).
        otherwise(F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

##Sales and Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (F.col("sls_price").isNull()) | (F.col("sls_price") <= 0),
            F.when(
                F.col("sls_quantity") != 0,
                F.col("sls_sales") / F.col("sls_quantity")
            ).otherwise(None)
        ).otherwise(F.col("sls_price"))
    )
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Checks Dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("workspace.silver.crm_sale")

## Sanity Checks Silver Table 

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sale LIMIT 10